# 🍄 Lab 2: Mushroom Survival — Would You Make It as a Forager?

You're lost in a forest. You're hungry. You find a mushroom. 🍄

**Do you eat it?**

Some mushrooms are delicious. Some will end your adventure permanently. ☠️ Today we build an AI that decides: **edible or poisonous?** — and we learn why *some mistakes matter much more than others*.

## 📚 What You'll Learn
- Turning text categories into numbers (encoding) 🔢
- Training a **Logistic Regression** classifier
- **Accuracy, Precision, and Recall** — and when accuracy LIES 🤥
- The confusion matrix

💡 **NOTE:** The **Golden Question 🏆** at the end is a matter of life and death. Literally. 😄

📦 **Dataset:** `mushrooms.csv` — 8,124 mushrooms described by 22 physical features.

In [ ]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn

### 🛠️ Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print("🍄 Ready to forage!")

### 📂 Load the Data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/iiFadel/majal-ai-track-2026/main/labs/day1/data/mushrooms.csv')
print("📏 Shape:", df.shape)
df.head()

### 🤨 Wait... it's all letters?!

Yes! Each letter is a code. For example, in the `class` column:
- `e` = **edible** 😋
- `p` = **poisonous** ☠️

And `cap-color`: `n` = brown, `y` = yellow, `w` = white... Every feature describes something physical about the mushroom (cap shape, odor, gill size...).

In [ ]:
# How many edible vs poisonous do we have?
print(df['class'].value_counts())

df['class'].value_counts().plot(kind='bar', color=['seagreen', 'firebrick'], figsize=(6, 4))
plt.title('🍄 Edible (e) vs Poisonous (p)')
plt.xticks(rotation=0)
plt.show()

# 👍 Good news: the dataset is BALANCED (roughly 50/50). Remember from theory why that matters!

### 🎯 Mini Exercise 1.1
Use `.value_counts()` to find: what is the most common **odor** in the dataset? (`odor` column — `n` means no smell, `f` means foul...) 👃

In [ ]:
# TO-DO: count the values of the 'odor' column


### 🔢 Section 2: Computers Don't Eat Letters — Encoding

Machine learning models only understand **numbers**. So we convert:

1. **The label:** poisonous → `1`, edible → `0`. (Poisonous is our "positive" class — the thing we're trying to detect, like "spam" in the theory!)
2. **The features:** we use **one-hot encoding** — each category becomes its own column of 0s and 1s. `pd.get_dummies()` does it in one line. ✨

In [ ]:
# 1) Encode the label: poisonous = 1 (positive class), edible = 0
y = (df['class'] == 'p').astype(int)

# 2) One-hot encode the features
X = pd.get_dummies(df.drop(columns=['class']))

print(f"🔢 We went from 22 letter-columns to {X.shape[1]} number-columns!")
X.head()

### 🤖 Section 3: Train the Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print(f"🎓 Trained on {len(X_train)} mushrooms")
print(f"🍄 Now testing on {len(X_test)} mushrooms the model has NEVER seen...")

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {acc * 100:.2f}%")

### 😲 That accuracy is amazing! But wait...

Accuracy tells us how often the model is right **overall**. But in a life-or-death problem, we need to know exactly **WHERE the mistakes are**. That's the job of the **confusion matrix**. 🔍

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Edible 😋', 'Poisonous ☠️'],
            yticklabels=['Edible 😋', 'Poisonous ☠️'])
plt.xlabel('Model predicted...')
plt.ylabel('The truth was...')
plt.title('🍄 Confusion Matrix')
plt.show()

print("Reading guide:")
print("⬆️ Top-right    = model said POISONOUS, truth was EDIBLE  → False Positive (wasted a good meal 😢)")
print("⬇️ Bottom-left  = model said EDIBLE, truth was POISONOUS → False Negative (☠️ DANGER ☠️)")

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"🎯 Precision: {precision * 100:.2f}%  → when the model shouts 'POISONOUS!', how often is it right?")
print(f"🕸️ Recall:    {recall * 100:.2f}%  → of ALL truly poisonous mushrooms, how many did it catch?")

### 🎯 Mini Exercise 3.1 — The Forager's Dilemma
Look at the confusion matrix. How many mushrooms would this model have told you to EAT... that were actually poisonous? (That's the bottom-left number.) Would you trust this model with your life? ✍️ Write your thoughts:

*Your answer here:* ...

### 🧪 Section 4: Make It Harder! 😈

Our model gets to see **22 features** — basically a full lab report on each mushroom. But a real forager in the forest can only check a few things quickly. What if the model could only see the **cap color and cap shape**?

In [ ]:
# Only 2 features this time!
X_small = pd.get_dummies(df[['cap-color', 'cap-shape']])

Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_small, y, test_size=0.2, random_state=42)

small_model = LogisticRegression(max_iter=1000)
small_model.fit(Xs_train, ys_train)
ys_pred = small_model.predict(Xs_test)

print(f"✅ Accuracy with ALL 22 features: {acc * 100:.2f}%")
print(f"😬 Accuracy with only 2 features: {accuracy_score(ys_test, ys_pred) * 100:.2f}%")
print(f"🕸️ Recall with only 2 features:   {recall_score(ys_test, ys_pred) * 100:.2f}%  ← how many poisonous ones we now MISS!")

### 🎯 Mini Exercise 4.1
Foragers say **smell** is one of the best poison indicators. Train a model using **only the `odor` column**. How does it compare to using cap color + shape? 👃

In [ ]:
# TO-DO: repeat Section 4, but with X_small = pd.get_dummies(df[['odor']])
# Print the accuracy AND the recall. Surprised? 😮


## The GOLDEN Question 🏆

Two mushroom-classifier apps are on the app store:

- **App A:** Precision 99%, Recall 90%
- **App B:** Precision 90%, Recall 99%

(Remember: "positive" = poisonous.)

**Which app do you install before your camping trip, and why? What does the other app's weakness mean in real life?** ⛺🍄

*Hint: which app misses fewer poisonous mushrooms?*